# NB00 · Validar fuentes (Spain Utilities Pulse)

**Primer paso del pipeline.** Comprueba que las APIs responden y devuelven datos desde **2024-01-01** hasta hoy. No escribe nada en el Lakehouse: solo muestra, por fuente, si la lectura fue bien, cuántas filas trae y el rango de fechas.

| Fuente | Qué se prueba |
|---|---|
| REE REData | Demanda eléctrica diaria (se pide por años para no superar los límites de la API) |
| ENTSOG | Flujo físico diario de Enagás, mes a mes |
| GIE AGSI+ | Almacenamiento subterráneo de gas de España (con paginación) |

- **MITECO no se prueba aquí:** es un zip con una base Access y se descarga y valida directamente en NB01.
- Este notebook es **informativo**: si una fuente falla, lo registra en la tabla de resultados pero no lanza error. Los notebooks que sí detienen el pipeline son NB01, NB02 y NB03.
- Para AGSI+ hace falta una clave gratuita (registro en [agsi.gie.eu](https://agsi.gie.eu)). Guárdala en `Files/config/utilities_config.json` con el contenido `{"agsi_key": "TU_CLAVE"}`. Si no existe, esa prueba se omite.

**Resultado de la ejecución del 23 nov 2026**

| Fuente | OK | Filas | Desde | Hasta | Nota |
|---|---|---|---|---|---|
| REE REData demanda diaria | ✔ | 1.057 | 2024-01-01 | 2026-11-22 | |
| ENTSOG flujo físico Enagás | ✔ | 20.957 | 2024-01-01 | 2026-11-23 | 15 puntos; ningún mes vacío |
| GIE AGSI+ almacenamiento ES | ✔ | 1.056 | 2024-01-01 | 2026-11-21 | |

In [ ]:
import json, os, datetime as dt
import requests
import pandas as pd

START = "2024-01-01"
END = dt.date.today().isoformat()
TIMEOUT = 60
resultados = []

def registrar(fuente, ok, filas=None, desde=None, hasta=None, nota=""):
    resultados.append({"fuente": fuente, "ok": ok, "filas": filas, "desde": desde, "hasta": hasta, "nota": nota})

print(f"Ventana de prueba: {START} -> {END}")

In [ ]:
# REE REData: demanda diaria (se pide por trozos de un año para no exceder límites)
def ree_demanda(ini, fin):
    url = "https://apidatos.ree.es/es/datos/demanda/evolucion"
    p = {"start_date": f"{ini}T00:00", "end_date": f"{fin}T23:59", "time_trunc": "day"}
    r = requests.get(url, params=p, timeout=TIMEOUT)
    r.raise_for_status()
    return r.json()

try:
    filas = []
    for ini, fin in [("2024-01-01", "2024-12-31"), ("2025-01-01", "2025-12-31"), ("2026-01-01", END)]:
        j = ree_demanda(ini, fin)
        for serie in j.get("included", []):
            for v in serie["attributes"]["values"]:
                filas.append({"serie": serie["attributes"]["title"], "fecha": v["datetime"][:10], "valor": v["value"]})
    df = pd.DataFrame(filas)
    display(df.head())
    registrar("REE REData demanda diaria", True, len(df), df["fecha"].min(), df["fecha"].max())
except Exception as e:
    registrar("REE REData demanda diaria", False, nota=str(e)[:200])

In [ ]:
# ENTSOG: flujo físico diario de Enagas, por meses desde START hasta hoy
URL_ENTSOG = "https://transparency.entsog.eu/api/v1/operationaldatas"

def entsog_tramo(ini, fin):
    r = requests.get(URL_ENTSOG, params={"indicator": "Physical Flow", "periodType": "day",
                     "operatorLabel": "Enagas", "from": ini, "to": fin,
                     "limit": -1, "timezone": "CET"}, timeout=TIMEOUT)
    r.raise_for_status()
    j = r.json()
    clave = next((k for k, v in j.items() if isinstance(v, list)), None)
    return pd.DataFrame(j[clave]) if clave else pd.DataFrame()

try:
    inicios = [d.date() for d in pd.date_range(START, END, freq="MS")]
    trozos, vacios = [], []
    for ini in inicios:
        fin = min((pd.Timestamp(ini) + pd.offsets.MonthBegin(1)).date(), dt.date.fromisoformat(END))
        d = entsog_tramo(ini.isoformat(), fin.isoformat())
        (trozos if len(d) else vacios).append(d if len(d) else ini.isoformat())
    df = pd.concat(trozos, ignore_index=True).drop_duplicates(["periodFrom", "pointKey", "directionKey"])
    print("Puntos distintos:", df["pointKey"].nunique(), "| meses sin datos:", vacios)
    display(df[["periodFrom", "pointKey", "pointLabel", "directionKey", "value", "unit"]].head())
    registrar("ENTSOG flujo físico Enagas", True, len(df), df["periodFrom"].min()[:10], df["periodFrom"].max()[:10],
              f"{df['pointKey'].nunique()} puntos; meses vacíos: {len(vacios)}")
except Exception as e:
    registrar("ENTSOG flujo físico Enagas", False, nota=str(e)[:200])

In [ ]:
# GIE AGSI+: almacenamiento subterráneo de España
cfg_path = "/lakehouse/default/Files/config/utilities_config.json"
if os.path.exists(cfg_path):
    key = json.load(open(cfg_path))["agsi_key"]
    try:
        filas, page = [], 1
        while True:
            r = requests.get("https://agsi.gie.eu/api", params={"country": "ES", "from": START, "to": END, "size": 300, "page": page},
                             headers={"x-key": key}, timeout=TIMEOUT)
            r.raise_for_status()
            j = r.json()
            filas += j.get("data", [])
            if page >= int(j.get("last_page", 1)):
                break
            page += 1
        df = pd.DataFrame(filas)
        display(df.head())
        registrar("GIE AGSI+ almacenamiento ES", len(df) > 0, len(df), df["gasDayStart"].min(), df["gasDayStart"].max())
    except Exception as e:
        registrar("GIE AGSI+ almacenamiento ES", False, nota=str(e)[:200])
else:
    registrar("GIE AGSI+ almacenamiento ES", False, nota="omitida: falta utilities_config.json con agsi_key")

In [ ]:
display(pd.DataFrame(resultados))